# 04. 신호등 API 전처리

공공데이터포털 `전국신호등표준데이터` API 전체를 수집하고,
229개 행정구역 기준으로 지역명을 통일한 뒤 동일 설치지점 중복을 제거합니다.

**출력**
- `../data_processed/traffic_light_count_final.csv`

> API에 데이터가 없는 지역은 시설 0개로 해석하지 않습니다.

In [1]:
import math
import time
import requests
import pandas as pd
from urllib.parse import unquote

URL = "https://api.data.go.kr/openapi/tn_pubr_public_traffic_light_api"

ENCODING_KEY = "dJ86KOJSsPyb6606Leb2zS7oit1l%2BHE1Vo2nmwAzyHvB7I1UdP%2B0guI2VeyfG9zFcQS7pvg2Ml11qoh8LcRIoA%3D%3D"
service_key = unquote(ENCODING_KEY)


## 1) API 전체 수집

In [5]:
params = {
    "serviceKey": service_key,
    "pageNo": 1,
    "numOfRows": 100,
    "type": "json"
}

res = requests.get(URL, params=params, timeout=30)
res.raise_for_status()

data = res.json()
total_count = int(data["body"]["totalCount"])

print("전체 건수:", total_count)

전체 건수: 98899


In [6]:
num_rows = 1000
total_pages = math.ceil(total_count / num_rows)
all_records = []

for page in range(1, total_pages + 1):
    params = {
        "serviceKey": service_key,
        "pageNo": page,
        "numOfRows": num_rows,
        "type": "json"
    }

    response = requests.get(URL, params=params, timeout=30)
    response.raise_for_status()

    records = (
        response.json()["body"]
        .get("items", {})
        .get("item", [])
    )

    if isinstance(records, dict):
        records = [records]

    all_records.extend(records)

    if page % 10 == 0 or page == total_pages:
        print(f"{page}/{total_pages} 완료 - 누적 {len(all_records)}건")

    time.sleep(0.05)

traffic_light = pd.DataFrame(all_records)
print("수집 완료:", traffic_light.shape)
display(traffic_light.head())

10/99 완료 - 누적 10000건
20/99 완료 - 누적 20000건
30/99 완료 - 누적 30000건
40/99 완료 - 누적 40000건
50/99 완료 - 누적 50000건
60/99 완료 - 누적 60000건
70/99 완료 - 누적 70000건
80/99 완료 - 누적 80000건
90/99 완료 - 누적 90000건
99/99 완료 - 누적 98899건
수집 완료: (98899, 34)


,ctprvnNm,signguNm,roadKnd,roadRouteNo,roadRouteNm,roadRouteDrc,rdnmadr,lnmadr,latitude,longitude,...,flashingLightCloseHhmm,fnctngSgngnrYn,remndrIdctYn,sondSgngnrYn,drcbrdSn,institutionNm,phoneNumber,referenceDate,insttCode,insttNm
0,서울특별시,동작구,시도,3119001,등용로,2,서울특별시 동작구 등용로 127,서울특별시 동작구 대방동 502,37.51255874,126.9309884,...,00:00,N,N,Y,,서울특별시 동작구청,02-820-1522,2026-08-26,3190000,서울특별시 동작구
1,서울특별시,동작구,특별시도,90,사당로,2,서울특별시 동작구 사당로 183,서울특별시 동작구 사당동 219-7,37.48601807,126.9691465,...,00:00,N,Y,Y,,서울특별시 동작구청,02-820-1522,2026-08-26,3190000,서울특별시 동작구
2,서울특별시,동작구,시도,3118027,대림로,2,서울특별시 동작구 대림로 61,서울특별시 동작구 신대방동 686,37.48853714,126.9066374,...,00:00,N,Y,Y,,서울특별시 동작구청,02-820-1522,2026-08-26,3190000,서울특별시 동작구
3,서울특별시,동작구,시도,3118027,대림로,2,서울특별시 동작구 대림로 54,서울특별시 동작구 신대방동 687-15,37.48830386,126.907637,...,00:00,Y,N,N,,서울특별시 동작구청,02-820-1522,2026-08-26,3190000,서울특별시 동작구
4,서울특별시,동작구,특별시도,21,시흥대로,1,서울특별시 동작구 시흥대로 638,서울특별시 동작구 신대방동 685-1,37.48884933,126.9067419,...,00:00,N,Y,Y,,서울특별시 동작구청,02-820-1522,2026-08-26,3190000,서울특별시 동작구


## 2) 지역명 229개 기준으로 정규화

In [7]:
PARENT_CITIES = [
    "수원시", "성남시", "안양시", "부천시",
    "안산시", "고양시", "용인시",
    "천안시", "청주시", "전주시",
    "포항시", "창원시"
]
GWANGJU_GU = {"동구", "서구", "남구", "북구", "광산구"}

def normalize_api_region(df):
    out = df.copy()

    out["ctprvnNm"] = out["ctprvnNm"].astype(str).str.strip()
    out["signguNm"] = out["signguNm"].astype(str).str.strip()

    out["SIDO_NM"] = out["ctprvnNm"]
    out["SIGUNGU_NM"] = out["signguNm"]

    # 데이터에 나타나는 통합 행정명칭 보정
    mask = out["SIDO_NM"] == "전남광주통합특별시"
    out.loc[
        mask & out["SIGUNGU_NM"].isin(GWANGJU_GU),
        "SIDO_NM"
    ] = "광주광역시"
    out.loc[
        mask & ~out["SIGUNGU_NM"].isin(GWANGJU_GU),
        "SIDO_NM"
    ] = "전라남도"

    # 일반시 아래 구 → 229개 기준 상위 시
    compact = out["SIGUNGU_NM"].str.replace(" ", "", regex=False)
    for city in PARENT_CITIES:
        city_mask = compact.str.startswith(city, na=False)
        out.loc[city_mask, "SIGUNGU_NM"] = city

    # 세종
    sejong_mask = (
        (out["SIDO_NM"] == "세종특별자치시") &
        (out["SIGUNGU_NM"].isin(["없음", "세종특별자치시"]))
    )
    out.loc[sejong_mask, "SIGUNGU_NM"] = "세종시"

    return out

In [8]:
tl = normalize_api_region(traffic_light)

mapping = pd.read_csv(
    "../data_raw/sgis_sigungu_252_to_229_mapping.csv"
)

region_229 = (
    mapping[["SIDO_NM", "sgg_nm_229"]]
    .drop_duplicates()
    .rename(columns={"sgg_nm_229": "SIGUNGU_NM"})
)

coverage = (
    region_229
    .merge(
        tl[["SIDO_NM", "SIGUNGU_NM"]].drop_duplicates(),
        on=["SIDO_NM", "SIGUNGU_NM"],
        how="left",
        indicator=True
    )
)

print("전국 기준 지역:", len(region_229))
print("신호등 API 데이터 있는 지역:",
      (coverage["_merge"] == "both").sum())
print("신호등 API 데이터 없는 지역:",
      (coverage["_merge"] == "left_only").sum())

전국 기준 지역: 229
신호등 API 데이터 있는 지역: 87
신호등 API 데이터 없는 지역: 142


## 3) 동일 설치지점 중복 제거 및 지역별 집계

In [9]:
# 지역 + 관리번호 + 좌표가 모두 같으면 동일 설치지점으로 간주
tl_unique = tl.drop_duplicates(
    subset=[
        "SIDO_NM",
        "SIGUNGU_NM",
        "tfclghtManageNo",
        "latitude",
        "longitude"
    ]
).copy()

traffic_light_count_final = (
    tl_unique
    .groupby(["SIDO_NM", "SIGUNGU_NM"])
    .size()
    .reset_index(name="traffic_light_count")
)

print("원본 API 행:", len(tl))
print("중복 정리 후 설치지점:", len(tl_unique))
print("데이터 제공 지역:", len(traffic_light_count_final))
display(traffic_light_count_final.head())

원본 API 행: 98899
중복 정리 후 설치지점: 96271
데이터 제공 지역: 87


,SIDO_NM,SIGUNGU_NM,traffic_light_count
0,강원특별자치도,강릉시,290
1,강원특별자치도,동해시,1154
2,강원특별자치도,삼척시,61
3,강원특별자치도,인제군,32
4,강원특별자치도,정선군,16


In [10]:
traffic_light_count_final.to_csv(
    "../data_processed/traffic_light_count_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

저장 완료
